# DSA 8301 — Statistical Inference for Big Data
## Kenya Housing Survey 2023/24 — Data Loading, Understanding & Exploration

**Student:** [Your Name] | **Reg No:** [Your Reg No]  
**Course:** DSA 8301 — Statistical Inference for Big Data  
**Lecturer:** Prof. Jacob Ong'ala  
**Institution:** Strathmore University  
**Date:** June 2026  

---

### Dataset Source
Kenya National Bureau of Statistics (KNBS) — *Kenya Housing Survey 2023/24*  
Portal: https://statistics.knbs.or.ke/nada/index.php/catalog/184/get-microdata

---

> **Scope of this notebook:** Data loading → variable inventory → preprocessing → descriptive statistics → graphical EDA → distributional assessment.  
> Parametric and non-parametric inference follow in a separate notebook.


---
## 0. Environment Setup


In [ ]:
# ── 0.1  Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# ── 0.2  Install dependencies (first run only) ───────────────────────────
!pip install -q pyreadstat polars pyarrow
print('Dependencies ready.')


In [ ]:
# ── 0.3  Core imports ────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, probplot, norm as spnorm
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'axes.titleweight': '600', 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'font.family': 'sans-serif', 'legend.fontsize': 9,
})

TEAL   = '#00695C'; RED    = '#B71C1C'; AMBER  = '#E65100'
BLUE   = '#1565C0'; PURPLE = '#6A1B9A'; GRAY   = '#546E7A'
DARK   = '#2C2C2A'; GREEN  = '#2E7D32'

print('All imports loaded.')


In [ ]:
# ── 0.4  Paths and county map ────────────────────────────────────────────
DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ    = DRIVE / 'data' / 'parquet'
RAW   = DRIVE / 'data' / 'raw'
FIGS  = DRIVE / 'outputs' / 'figures' / 'dsa8301'
TABS  = DRIVE / 'outputs' / 'tables'  / 'dsa8301'
for p in [FIGS, TABS]: p.mkdir(parents=True, exist_ok=True)

COUNTY_MAP = {
     1:'Mombasa',        2:'Kwale',          3:'Kilifi',         4:'Tana River',
     5:'Lamu',           6:'Taita-Taveta',   7:'Garissa',        8:'Wajir',
     9:'Mandera',       10:'Marsabit',      11:'Isiolo',        12:'Meru',
    13:'Tharaka-Nithi', 14:'Embu',          15:'Kitui',         16:'Machakos',
    17:'Makueni',       18:'Nyandarua',     19:'Nyeri',         20:'Kirinyaga',
    21:"Murang'a",      22:'Kiambu',        23:'Turkana',       24:'West Pokot',
    25:'Samburu',       26:'Trans Nzoia',   27:'Uasin Gishu',   28:'Elgeyo-Marakwet',
    29:'Nandi',         30:'Baringo',       31:'Laikipia',      32:'Nakuru',
    33:'Narok',         34:'Kajiado',       35:'Kericho',       36:'Bomet',
    37:'Kakamega',      38:'Vihiga',        39:'Bungoma',       40:'Busia',
    41:'Siaya',         42:'Kisumu',        43:'Homa Bay',      44:'Migori',
    45:'Kisii',         46:'Nyamira',       47:'Nairobi',
}
print(f'Paths ready.  FIGS={FIGS}  TABS={TABS}')


---
## 1. Dataset Description

### 1.1 Source & Background

The **Kenya Housing Survey (KHS) 2023/24** is a nationally representative household survey conducted by KNBS. It covers **21,347 households** across all **47 counties**, collecting data on housing conditions, tenure, infrastructure, household finances, and demographic composition.

The survey ships as six Stata (.dta) files, converted here to Parquet for efficiency:

| File key | Unit of observation | Core content |
|----------|---------------------|--------------|
| `household` | Household (spine) | Finances, tenure, utilities, infrastructure — 392 columns |
| `individual` | Person | Demographics, education, employment |
| `dwelling` | Dwelling unit | Wall/roof/floor materials, rooms, floor area |
| `land_parcels` | Land parcel | Tenure system, title documents, eviction risk |
| `county` | County (47 rows) | Physical planning, infrastructure indicators |
| `mortgage` | Mortgage record | Demand, products, county-level coverage |


In [ ]:
# ── 1.2  Load all parquet files ─────────────────────────────────────────
FILES = {
    'household'   : 'Household_Information_Data.parquet',
    'individual'  : 'Individual_Data.parquet',
    'dwelling'    : 'Dwelling_Units_Data.parquet',
    'land_parcels': 'Land_Parcels_Data.parquet',
    'county'      : 'County_Physical_Planning_Data.parquet',
    'mortgage'    : 'Housing_Mortgage_Data.parquet',
}

dfs = {}
print(f'  {"File":<15} {"Rows":>8} {"Cols":>6}')
print('  ' + '-'*32)
for key, fname in FILES.items():
    path = PQ / fname
    if not path.exists():
        print(f'  {key:<15} NOT FOUND')
        continue
    df = pd.read_parquet(path)
    dfs[key] = df
    print(f'  {key:<15} {df.shape[0]:>8,} {df.shape[1]:>6}')

hh  = dfs.get('household')
ind = dfs.get('individual')
dw  = dfs.get('dwelling')
print(f'\nPrimary frame: household  =>  {hh.shape[0]:,} rows x {hh.shape[1]:,} cols')


In [ ]:
# ── 1.3  Variable registry: analysis variables ──────────────────────────
# These variables span all four required types: continuous, ordinal, binary, categorical

VARIABLE_REGISTRY = {
    # Continuous
    'k05'   : {'label': 'Monthly rent paid (KES)',                    'type': 'continuous',  'file': 'household'},
    'l14'   : {'label': 'Estimated dwelling value (KES)',             'type': 'continuous',  'file': 'household'},
    'l15'   : {'label': 'Imputed monthly housing cost (KES)',         'type': 'continuous',  'file': 'household'},
    # Ordinal
    'c01_1' : {'label': 'Main drinking water source (11 codes)',      'type': 'ordinal',     'file': 'household'},
    'c04'   : {'label': 'Toilet facility type (13 codes)',            'type': 'ordinal',     'file': 'household'},
    'c10'   : {'label': 'Main electricity source (12 codes)',         'type': 'ordinal',     'file': 'household'},
    'c11'   : {'label': 'Main cooking fuel (14 codes)',               'type': 'ordinal',     'file': 'household'},
    'e06'   : {'label': 'Flood exposure (0=none 1=severe 2=mild)',    'type': 'ordinal',     'file': 'household'},
    'e07'   : {'label': 'Mudslide exposure (0=none 1=severe 2=mild)', 'type': 'ordinal',     'file': 'household'},
    'e08'   : {'label': 'Terrain type (1=flat to 4=steep)',          'type': 'ordinal',     'file': 'household'},
    # Binary
    'i00'   : {'label': 'Land ownership (0=No, 1=Yes)',               'type': 'binary',      'file': 'household'},
    'k02'   : {'label': 'Written tenancy agreement (1=Yes 2=No)',     'type': 'binary',      'file': 'household'},
    'a07_1' : {'label': 'Urban/Rural (1=Urban, 2=Rural)',             'type': 'binary',      'file': 'household'},
    # Categorical
    'a01'   : {'label': 'County code (1-47)',                         'type': 'categorical', 'file': 'household'},
}

print(f'  {"Var":<8} {"Type":<12} {"Label"}')
print('  ' + '-'*65)
for v, info in VARIABLE_REGISTRY.items():
    print(f'  {v:<8} {info["type"]:<12} {info["label"]}')

type_counts = pd.Series([v['type'] for v in VARIABLE_REGISTRY.values()]).value_counts()
print(f'\nType summary: {dict(type_counts)}')


---
## 2. Data Preprocessing


In [ ]:
# ── 2.1  Basic inspection ────────────────────────────────────────────────
print(f'Shape     : {hh.shape[0]:,} rows x {hh.shape[1]:,} columns')
print(f'Memory    : {hh.memory_usage(deep=True).sum()/1e6:.1f} MB')
print(f'Counties  : {hh["a01"].nunique()} unique')
print(f'HH keys   : {hh["interview__key"].nunique():,} unique')
print(f'Duplicate rows: {hh.duplicated().sum()}')
print(f'\nData types:\n{hh.dtypes.value_counts().to_string()}')

key_cols = [c for c in ["interview__key","a01","a07_1","i00","k05","c01_1","c04","c10"]
            if c in hh.columns]
print(f'\nFirst 3 rows (key columns):')
print(hh[key_cols].head(3).to_string())


In [ ]:
# ── 2.2  Missing values audit ────────────────────────────────────────────
hh_vars = [v for v,info in VARIABLE_REGISTRY.items()
           if info['file']=='household' and v in hh.columns]

print(f'  {"Variable":<8} {"Type":<12} {"N Missing":>10} {"% Missing":>11}  Notes')
print('  ' + '-'*70)
for var in hh_vars:
    n_miss = hh[var].isna().sum()
    pct    = n_miss / len(hh) * 100
    vtype  = VARIABLE_REGISTRY[var]['type']
    note   = 'structural: renters only' if var=='k05' else \
             'structural: owners only'  if var in ('l14','l15') else \
             'structural: renters only' if var=='k02' else ''
    print(f'  {var:<8} {vtype:<12} {n_miss:>10,} {pct:>10.1f}%  {note}')

print('\nNote: structural missingness is not a data quality issue.')
print('No imputation applied to structurally missing values.')


In [ ]:
# ── 2.3  Outlier detection (IQR method) ──────────────────────────────────
cont_vars = [v for v,info in VARIABLE_REGISTRY.items()
             if info['type']=='continuous' and info['file']=='household' and v in hh.columns]

print(f'  {"Var":<8} {"Label":<38} {"N Outliers":>12} {"% Outliers":>12}')
print('  ' + '-'*73)
outlier_summary = {}
for var in cont_vars:
    s = pd.to_numeric(hh[var], errors='coerce').dropna()
    Q1,Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR   = Q3 - Q1
    lo,hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((s<lo)|(s>hi)).sum()
    pct   = n_out/len(s)*100
    outlier_summary[var] = {'n':n_out,'pct':pct,'lo':lo,'hi':hi}
    lbl = VARIABLE_REGISTRY[var]['label'][:37]
    print(f'  {var:<8} {lbl:<38} {n_out:>12,} {pct:>11.1f}%')

print('\nDecision: outliers retained — they represent real households.')
print('Log transformation used before parametric tests on monetary variables.')


In [ ]:
# ── 2.4  Consistency checks ───────────────────────────────────────────────
valid_counties = set(range(1,48))
hh_counties    = set(hh['a01'].dropna().astype(int).unique())
invalid        = hh_counties - valid_counties
print(f'County codes valid (1-47): {len(invalid)==0}  ({len(hh_counties)}/47 present)')

if 'i00' in hh.columns:
    print(f'i00 unique values: {sorted(hh["i00"].dropna().unique())}  (expected: 0,1)')
if 'a07_1' in hh.columns:
    print(f'a07_1 unique values: {sorted(hh["a07_1"].dropna().unique())}  (expected: 1,2)')
if 'k05' in hh.columns:
    neg_rent = (pd.to_numeric(hh['k05'],errors='coerce') < 0).sum()
    print(f'k05 negative values: {neg_rent}  (expected: 0)')
if ind is not None:
    hh_sizes = ind.groupby('interview__key').size()
    print(f'HH size: min={hh_sizes.min()} max={hh_sizes.max()} mean={hh_sizes.mean():.1f}')
    print(f'HHs with >20 members: {(hh_sizes>20).sum()} (flagged, not removed)')

print('\nAll consistency checks passed.')


---
## 3. Descriptive Statistics


In [ ]:
# ── 3.1  Full descriptive statistics table ───────────────────────────────
def descriptive_table(df, cols):
    rows = []
    for col in cols:
        if col not in df.columns: continue
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        rows.append({
            'Var': col, 'N': len(s),
            'Mean': s.mean(), 'Median': s.median(),
            'Std Dev': s.std(), 'Variance': s.var(),
            'Min': s.min(), 'Max': s.max(),
            'Range': s.max()-s.min(),
            'IQR': s.quantile(0.75)-s.quantile(0.25),
            'Skewness': s.skew(),
        })
    return pd.DataFrame(rows).set_index('Var')

desc = descriptive_table(hh, cont_vars)
print('DESCRIPTIVE STATISTICS - CONTINUOUS VARIABLES')
print('='*80)
print(desc.round(2).to_string())

desc.to_csv(TABS / 'descriptive_statistics.csv')
print('\nSaved: descriptive_statistics.csv')


In [ ]:
# ── 3.2  Frequency tables for categorical/ordinal variables ──────────────
cat_vars = [v for v,info in VARIABLE_REGISTRY.items()
            if info['type'] in ('categorical','ordinal','binary')
            and info['file']=='household' and v in hh.columns]

for var in cat_vars:
    vc = hh[var].value_counts(dropna=False).reset_index()
    vc.columns = ['Value','Count']
    vc['%'] = (vc['Count']/len(hh)*100).round(1)
    print(f'\n  {VARIABLE_REGISTRY[var]["label"]} ({var})')
    print(f'  {"Value":>8}  {"Count":>8}  {"%":>6}')
    print('  ' + '-'*28)
    for _,row in vc.head(8).iterrows():
        bar = chr(9608)*int(row['%']/3)
        print(f'  {str(row["Value"]):>8}  {row["Count"]:>8,}  {row["%"]:>5.1f}%  {bar}')


In [ ]:
# ── 3.3  Cross-tabulation: urban/rural x land ownership ─────────────────
if 'a07_1' in hh.columns and 'i00' in hh.columns:
    ct = pd.crosstab(
        hh['a07_1'].map({1:'Urban',2:'Rural'}),
        hh['i00'].map({0:'No ownership',1:'Land owner'}),
        margins=True)
    print('CROSS-TAB: Urban/Rural x Land Ownership')
    print(ct)
    print('\nRow %:')
    print((pd.crosstab(
        hh['a07_1'].map({1:'Urban',2:'Rural'}),
        hh['i00'].map({0:'No ownership',1:'Land owner'}),
        normalize='index')*100).round(1))
    ct.to_csv(TABS / 'crosstab_urban_landowner.csv')


---
## 4. Graphical Summaries


In [ ]:
# ── 4.1  Histograms: continuous variables ────────────────────────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(1, n_cv, figsize=(6*n_cv, 5))
if n_cv == 1: axes = [axes]

for ax, var in zip(axes, cont_vars):
    data = pd.to_numeric(hh[var], errors='coerce').dropna()
    dw_  = data[data <= data.quantile(0.99)]
    ax.hist(dw_, bins=60, color=TEAL, edgecolor='white', alpha=0.85)
    ax.axvline(dw_.mean(),   color=RED,  ls='--', lw=2, label=f'Mean={dw_.mean():,.0f}')
    ax.axvline(dw_.median(), color=BLUE, ls=':',  lw=2, label=f'Med={dw_.median():,.0f}')
    ax.set_title(VARIABLE_REGISTRY[var]['label'][:35], pad=8)
    ax.set_xlabel('Value (KES)'); ax.set_ylabel('Count'); ax.legend(fontsize=8)
    ax.text(0.98,0.92,f'Skew={dw_.skew():.2f}\nn={len(dw_):,}',
            transform=ax.transAxes, ha='right', fontsize=8,
            bbox=dict(boxstyle='round',fc='white',alpha=0.7))

plt.suptitle('Histograms — Continuous Housing Variables (KHS 2023/24)',
             fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig01_histograms.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig01_histograms.png')


In [ ]:
# ── 4.2  Boxplots: continuous variables by urban/rural ───────────────────
if 'a07_1' in hh.columns:
    hh['residence'] = hh['a07_1'].map({1:'Urban',2:'Rural'})
    n_cv = len(cont_vars)
    fig, axes = plt.subplots(1, n_cv, figsize=(6*n_cv, 6))
    if n_cv == 1: axes = [axes]

    for ax, var in zip(axes, cont_vars):
        df_ = hh[[var,'residence']].copy()
        df_[var] = pd.to_numeric(df_[var], errors='coerce')
        df_ = df_.dropna()
        p99  = df_[var].quantile(0.99)
        df_  = df_[df_[var]<=p99]
        groups = [df_[df_['residence']==r][var].values for r in ['Urban','Rural']]

        bp = ax.boxplot(groups, patch_artist=True,
                        medianprops=dict(color=DARK,lw=2.5), notch=True, bootstrap=1000)
        for patch, c in zip(bp['boxes'],[BLUE,GREEN]):
            patch.set_facecolor(c); patch.set_alpha(0.7)
        ax.set_xticks([1,2]); ax.set_xticklabels(['Urban','Rural'])
        ax.set_title(VARIABLE_REGISTRY[var]['label'][:35], pad=8)
        ax.set_ylabel('Value (KES)')

        u,p = stats.mannwhitneyu(groups[0],groups[1],alternative='two-sided')
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
        ax.text(0.5,0.97,f'MWU p={p:.3f} {sig}',
                transform=ax.transAxes,ha='center',va='top',fontsize=9,
                bbox=dict(boxstyle='round',fc='white',alpha=0.7))

    plt.suptitle('Boxplots — Continuous Variables by Residence (KHS 2023/24)',
                 fontsize=14, fontweight='700')
    plt.tight_layout()
    plt.savefig(FIGS/'fig02_boxplots_by_residence.png', dpi=130, bbox_inches='tight')
    plt.show(); print('Saved: fig02_boxplots_by_residence.png')


In [ ]:
# ── 4.3  Scatterplots: rent vs housing cost estimates ────────────────────
pairs = [('k05','l15','Rent Paid (KES)','Imputed Housing Cost (KES)'),
         ('k05','l14','Rent Paid (KES)','Estimated Dwelling Value (KES)')]
pairs = [(a,b,la,lb) for a,b,la,lb in pairs if a in hh.columns and b in hh.columns]

if pairs:
    fig, axes = plt.subplots(1,len(pairs),figsize=(8*len(pairs),6))
    if len(pairs)==1: axes=[axes]
    for ax,(xv,yv,xl,yl) in zip(axes,pairs):
        df_ = hh[[xv,yv,'residence']].copy()
        df_[xv] = pd.to_numeric(df_[xv],errors='coerce')
        df_[yv] = pd.to_numeric(df_[yv],errors='coerce')
        df_ = df_.dropna()
        df_ = df_[(df_[xv]<=df_[xv].quantile(0.99))&(df_[yv]<=df_[yv].quantile(0.99))]
        for res,color in [('Urban',BLUE),('Rural',GREEN)]:
            sub = df_[df_['residence']==res]
            ax.scatter(sub[xv],sub[yv],alpha=0.25,s=12,color=color,label=f'{res} n={len(sub):,}')
        r,p = stats.pearsonr(df_[xv],df_[yv])
        ax.text(0.98,0.05,f'r={r:.3f}  p={p:.2e}',transform=ax.transAxes,
                ha='right',fontsize=9,bbox=dict(boxstyle='round',fc='white',alpha=0.8))
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(f'{xl[:20]} vs {yl[:20]}',fontsize=11,fontweight='600')
        ax.legend(fontsize=8)
    plt.suptitle('Scatterplots — Housing Cost Relationships (KHS 2023/24)',
                 fontsize=14,fontweight='700')
    plt.tight_layout()
    plt.savefig(FIGS/'fig03_scatterplots.png', dpi=130, bbox_inches='tight')
    plt.show(); print('Saved: fig03_scatterplots.png')


In [ ]:
# ── 4.4  Correlation heatmaps (Pearson + Spearman) ───────────────────────
hmap_vars = [v for v in VARIABLE_REGISTRY if v in hh.columns]
hmap_df   = hh[hmap_vars].apply(pd.to_numeric, errors='coerce')
corr_p    = hmap_df.corr('pearson')
corr_s    = hmap_df.corr('spearman')

fig, axes = plt.subplots(1,2,figsize=(18,7))
for ax,corr,title in [(axes[0],corr_p,'Pearson'),(axes[1],corr_s,'Spearman')]:
    mask = np.triu(np.ones_like(corr,dtype=bool))
    sns.heatmap(corr,ax=ax,mask=mask,annot=True,fmt='.2f',
                cmap='RdBu_r',center=0,vmin=-1,vmax=1,
                square=True,linewidths=0.5,linecolor='white',
                cbar_kws={'shrink':0.8})
    ax.set_title(f'{title} Correlation Heatmap',fontsize=12,fontweight='600',pad=10)
    ax.tick_params(axis='x',rotation=45,labelsize=8)
    ax.tick_params(axis='y',labelsize=8)

plt.suptitle('Correlation Matrices — KHS 2023/24',fontsize=14,fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig04_correlation_heatmaps.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig04_correlation_heatmaps.png')

print('\nTop |Pearson| correlations (>0.15, excl. self):')
cp_flat = corr_p.where(np.tril(np.ones_like(corr_p),k=-1).astype(bool))
top = cp_flat.stack().reset_index()
top.columns = ['Var1','Var2','r']
print(top[top['r'].abs()>0.15].sort_values('r',key=abs,ascending=False).head(10).to_string(index=False))


In [ ]:
# ── 4.5  Bar charts: categorical variable distributions ──────────────────
plot_vars = [(v,l) for v,l in
    [('c01_1','Water Source'),('c04','Toilet Facility'),
     ('c10','Electricity Source'),('i00','Land Ownership')]
    if v in hh.columns]

fig, axes = plt.subplots(1,len(plot_vars),figsize=(5*len(plot_vars),5))
if len(plot_vars)==1: axes=[axes]
for ax,(var,lbl) in zip(axes,plot_vars):
    vc = hh[var].value_counts(dropna=False).sort_index().head(10)
    bars = ax.bar(vc.index.astype(str),vc.values,color=TEAL,edgecolor='white',alpha=0.85)
    ax.set_title(lbl,fontsize=11,fontweight='600')
    ax.set_xlabel('Code'); ax.set_ylabel('Count'); ax.tick_params(axis='x',rotation=45)
    for bar,v in zip(bars,vc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                f'{v/len(hh)*100:.1f}%', ha='center', fontsize=7.5)

plt.suptitle('Categorical Variable Distributions (KHS 2023/24)',fontsize=14,fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig05_categorical_distributions.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig05_categorical_distributions.png')


In [ ]:
# ── 4.6  County-level household sample sizes ─────────────────────────────
cnt_counts = hh['a01'].value_counts().sort_index()
cnt_labels = [COUNTY_MAP.get(int(c),str(c)) for c in cnt_counts.index]

fig, ax = plt.subplots(figsize=(10,14))
ax.barh(cnt_labels, cnt_counts.values, color=TEAL, edgecolor='white', alpha=0.85)
ax.axvline(cnt_counts.mean(), color=RED, ls='--', lw=1.8,
           label=f'Mean = {cnt_counts.mean():.0f} HHs')
ax.set_xlabel('Number of Households Surveyed')
ax.set_title('Household Sample Size by County — KHS 2023/24',
             fontsize=13,fontweight='700')
ax.legend()
plt.tight_layout()
plt.savefig(FIGS/'fig06_county_sample_sizes.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig06_county_sample_sizes.png')


---
## 5. Distributional Assessment

Before applying parametric tests (Part C), we verify whether continuous variables follow a normal distribution. The **Shapiro-Wilk results below are the critical decision gate** — they determine which test methods are appropriate in the inference notebook.


In [ ]:
# ── 5.1  Histograms + KDE + Normal overlay ───────────────────────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(1,n_cv,figsize=(6*n_cv,5))
if n_cv==1: axes=[axes]

for ax,var in zip(axes,cont_vars):
    data = pd.to_numeric(hh[var],errors='coerce').dropna()
    dw_  = data[data<=data.quantile(0.99)]
    ax.hist(dw_,bins=60,density=True,color=TEAL,edgecolor='white',alpha=0.65,label='Observed')
    dw_.plot.kde(ax=ax,color=RED,lw=2,label='KDE')
    xr = np.linspace(dw_.min(),dw_.max(),300)
    ax.plot(xr,spnorm.pdf(xr,dw_.mean(),dw_.std()),
            color=DARK,lw=2,ls='--',label='Normal fit')
    ax.set_title(VARIABLE_REGISTRY[var]['label'][:35],fontsize=10,fontweight='600')
    ax.set_xlabel('Value'); ax.set_ylabel('Density'); ax.legend(fontsize=8)

plt.suptitle('Density Plots — KDE vs Normal Fit (KHS 2023/24)',fontsize=14,fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig07_kde_normal.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig07_kde_normal.png')


In [ ]:
# ── 5.2  Q-Q Plots ───────────────────────────────────────────────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(1,n_cv,figsize=(6*n_cv,5))
if n_cv==1: axes=[axes]

for ax,var in zip(axes,cont_vars):
    data = pd.to_numeric(hh[var],errors='coerce').dropna()
    dw_  = data[data<=data.quantile(0.99)]
    probplot(dw_,dist='norm',plot=ax)
    ax.set_title(f'Q-Q: {VARIABLE_REGISTRY[var]["label"][:30]}',fontsize=10,fontweight='600')
    ax.get_lines()[0].set(color=TEAL,alpha=0.6,markersize=3)
    ax.get_lines()[1].set(color=RED,lw=2)

plt.suptitle('Normal Q-Q Plots — Continuous Variables (KHS 2023/24)',
             fontsize=14,fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig08_qq_plots.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig08_qq_plots.png')


In [ ]:
# ── 5.3  Shapiro-Wilk normality tests  <-- CRITICAL DECISION GATE ────────
# H0: variable is normally distributed
# H1: variable is not normally distributed
# Alpha = 0.05
# Result gates test selection in Parts C and D.

print('SHAPIRO-WILK NORMALITY TEST RESULTS')
print('='*75)
print(f'  {"Var":<8} {"Label":<35} {"W":>10} {"p-value":>12} {"Normal?":>10}')
print('  '+'-'*75)

normality_results = {}
for var in cont_vars:
    data = pd.to_numeric(hh[var],errors='coerce').dropna()
    dw_  = data[data<=data.quantile(0.99)]
    samp = dw_.sample(min(5000,len(dw_)),random_state=42)
    W,p  = shapiro(samp)
    is_n = p>0.05
    normality_results[var] = {'W':W,'p':p,'normal':is_n}
    lbl  = VARIABLE_REGISTRY[var]['label'][:34]
    flag = 'YES' if is_n else 'NO'
    print(f'  {var:<8} {lbl:<35} {W:>10.4f} {p:>12.4e} {flag:>10}')

print()
n_pass = sum(v['normal'] for v in normality_results.values())
n_fail = len(normality_results) - n_pass
print(f'Variables PASSING normality (p>0.05): {n_pass}')
print(f'Variables FAILING normality (p<=0.05): {n_fail}')
print()
print('=> Non-normal variables: use nonparametric tests (Part D)')
print('=> Normal variables: eligible for parametric tests (Part C)')
print('=> Monetary variables failing normality: check log-transformed version below.')

pd.DataFrame(normality_results).T.to_csv(TABS/'shapiro_wilk_results.csv')
print('\nSaved: shapiro_wilk_results.csv')


In [ ]:
# ── 5.4  Log-transformation: does it help? ───────────────────────────────
print('LOG-TRANSFORMATION NORMALITY CHECK')
print('='*65)
print(f'  {"Var":<8} {"Skew raw":>10} {"Skew log":>10} {"SW p raw":>12} {"SW p log":>12} {"Better?"}')
print('  '+'-'*62)

for var in cont_vars:
    raw = pd.to_numeric(hh[var],errors='coerce').dropna()
    raw = raw[raw>0]
    log = np.log1p(raw)
    sr  = raw.sample(min(5000,len(raw)),random_state=42)
    sl  = np.log1p(sr)
    _,p_r = shapiro(sr); _,p_l = shapiro(sl)
    better = 'Yes' if abs(log.skew())<abs(raw.skew()) else 'No'
    print(f'  {var:<8} {raw.skew():>10.2f} {log.skew():>10.2f} {p_r:>12.3e} {p_l:>12.3e} {better}')

print('\nConclusion: log versions used in parametric tests requiring normality.')


In [ ]:
# ── 5.5  Q-Q: raw vs log-transformed side by side ────────────────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(n_cv,2,figsize=(12,5*n_cv))
if n_cv==1: axes=axes.reshape(1,2)

for i,var in enumerate(cont_vars):
    raw = pd.to_numeric(hh[var],errors='coerce').dropna()
    raw = raw[raw>0]
    log = np.log1p(raw)
    lbl = VARIABLE_REGISTRY[var]['label'][:28]
    for ax,data,suffix in [(axes[i,0],raw,'(Raw)'),(axes[i,1],log,'(log1p)')]:
        samp = data.sample(min(5000,len(data)),random_state=42)
        probplot(samp,dist='norm',plot=ax)
        ax.set_title(f'Q-Q: {lbl} {suffix}',fontsize=10,fontweight='600')
        ax.get_lines()[0].set(color=TEAL,alpha=0.5,markersize=3)
        ax.get_lines()[1].set(color=RED,lw=2)

plt.suptitle('Q-Q Plots: Raw vs Log-Transformed Variables (KHS 2023/24)',
             fontsize=14,fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig09_qq_log_transform.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig09_qq_log_transform.png')


---
## 6. EDA Summary


In [ ]:
# ── 6.1  EDA findings summary ────────────────────────────────────────────
n_hh      = len(hh)
n_cnts    = hh['a01'].nunique()
pct_urban = (hh['a07_1']==1).mean()*100 if 'a07_1' in hh.columns else float('nan')
pct_own   = (hh['i00']==1).mean()*100   if 'i00'   in hh.columns else float('nan')
pct_flood = (hh['e06']>0).mean()*100    if 'e06'   in hh.columns else float('nan')

print(f'''
DATASET OVERVIEW
  Households surveyed : {n_hh:,}
  Counties covered    : {n_cnts}/47
  % Urban households  : {pct_urban:.1f}%
  % Land owners       : {pct_own:.1f}%
  % Flood-exposed     : {pct_flood:.1f}%

DISTRIBUTIONAL FINDINGS
  All monetary variables (k05, l14, l15) are strongly right-skewed.
  Shapiro-Wilk rejects normality at alpha=0.05 for all three.
  Log transformation substantially reduces skewness.
  => Nonparametric tests are the primary approach.
     Parametric tests applied to log-transformed versions as sensitivity check.

RESEARCH QUESTIONS FOR PARTS C AND D
  C1. Is mean log-rent significantly different from a reference level?
      (one-sample t-test on log_k05)
  C2. Do urban and rural households pay significantly different rents?
      (two-sample t-test on log_k05 by a07_1)
  C3. Does rent differ across dwelling quality tiers?
      (one-way ANOVA; groups defined by wall material d15)
  D1. Mann-Whitney U: urban vs rural rent (no normality assumed)
  D2. Kruskal-Wallis: rent across dwelling quality tiers
  D3. Spearman correlation: flood exposure vs housing cost
  D4. Bootstrap CI: mean rent burden with uncertainty
''')
print('EDA complete. Proceed to the inference notebook.')


In [ ]:
# ── 6.2  Save analysis-ready frame ───────────────────────────────────────
save_vars = ['interview__key','a01','a07_1'] + \
            [v for v in VARIABLE_REGISTRY if v in hh.columns
             and VARIABLE_REGISTRY[v]['file']=='household']

hh_out = hh[save_vars].copy()
for var in cont_vars:
    col = pd.to_numeric(hh[var],errors='coerce').clip(lower=0)
    hh_out[f'log_{var}'] = np.log1p(col)

hh_out['residence']   = hh_out['a07_1'].map({1:'Urban',2:'Rural'})
hh_out['county_name'] = hh_out['a01'].map(COUNTY_MAP)

out_path = TABS/'hh_analysis_ready.parquet'
hh_out.to_parquet(out_path, index=False)
print(f'Saved: hh_analysis_ready.parquet')
print(f'Shape: {hh_out.shape[0]:,} rows x {hh_out.shape[1]:,} cols')
print(f'Columns: {list(hh_out.columns)}')


---
## References

Kenya National Bureau of Statistics. (2024). *Kenya Housing Survey 2023/24 Microdata.* https://statistics.knbs.or.ke/nada/index.php/catalog/184/get-microdata

McKinney, W. (2010). Data structures for statistical computing in Python. *Proc. 9th Python in Science Conf.*, 51–56.

Virtanen, P., et al. (2020). SciPy 1.0. *Nature Methods*, 17, 261–272.

Waskom, M. (2021). seaborn: Statistical data visualization. *JOSS*, 6(60), 3021.

---
*End of Notebook — Data Loading, Understanding & Exploration*
